# Language-Guided Scene to Action

This notebook is an original starter project for Vision-Language-Action autonomous driving. It combines synthetic visual scene cues with a language command and returns a high-level driving action plus an explanation.

In [ ]:
import pandas as pd

scenes = [
    {
        "scene_id": "urban_001",
        "traffic_light": "red",
        "pedestrian_crossing": True,
        "lead_vehicle_distance_m": 14,
        "lane_clear": True,
        "command": "turn right after the crosswalk when safe",
    },
    {
        "scene_id": "urban_002",
        "traffic_light": "green",
        "pedestrian_crossing": False,
        "lead_vehicle_distance_m": 42,
        "lane_clear": True,
        "command": "continue straight through the intersection",
    },
    {
        "scene_id": "urban_003",
        "traffic_light": "green",
        "pedestrian_crossing": False,
        "lead_vehicle_distance_m": 8,
        "lane_clear": False,
        "command": "keep going in the current lane",
    },
    {
        "scene_id": "urban_004",
        "traffic_light": "yellow",
        "pedestrian_crossing": False,
        "lead_vehicle_distance_m": 22,
        "lane_clear": True,
        "command": "turn left at the next intersection",
    },
]

df = pd.DataFrame(scenes)
df

In [ ]:
def parse_intent(command):
    text = command.lower()
    if "left" in text:
        return "turn_left"
    if "right" in text:
        return "turn_right"
    if "straight" in text or "continue" in text or "keep" in text:
        return "keep_lane"
    return "keep_lane"


def recommend_action(scene):
    intent = parse_intent(scene["command"])

    if scene["traffic_light"] == "red":
        return "stop", "Traffic light is red, so stopping overrides the navigation command."

    if scene["pedestrian_crossing"]:
        return "yield", "A pedestrian is crossing, so the safest action is to yield."

    if scene["lead_vehicle_distance_m"] < 10:
        return "slow_down", "The lead vehicle is close, so reduce speed before executing the command."

    if not scene["lane_clear"]:
        return "slow_down", "The lane is not clear, so slow down and wait for a safer gap."

    if scene["traffic_light"] == "yellow":
        return "slow_down", "The light is yellow, so slow down before deciding whether to proceed."

    return intent, f"The scene is clear, so execute the language intent: {intent}."


df[["action", "explanation"]] = df.apply(lambda row: pd.Series(recommend_action(row)), axis=1)
df[["scene_id", "command", "action", "explanation"]]

## Next Step

Replace the synthetic scene dictionary with outputs from a detector, lane model, traffic light classifier, or segmentation model. The action policy can stay the same while perception becomes more realistic.